# Semana 11: análise exploratória E7, CKA entre o codificador pré-treinado e o ajustado

Segue o E7 de `prereg/plano_exploratorio_2.md`. **É exploratório**: não aplica categorias do pré-registro e não substitui nada da rodada confirmatória.

Para cada execução, passa as mesmas imagens pelo codificador pré-treinado e pelo codificador ajustado e acumula, por grupo de unidades (teste da fonte e cada evento do alvo) e por bloco do codificador (5, 11, 17 e 23, os que alimentam o pescoço), as somas do CKA linear. Grava um arquivo JSON por execução com o CKA de cada grupo e bloco. Execuções: uma do R1, que serve de conferência (o codificador do R1 é congelado, então o CKA tem de dar 1), e as cinco do R2 e as cinco do R3.

Conferência de cada modelo: na Bolívia, a margem da passada completa é comparada com a gravada na semana 6; acima de 1e-3, nada é gravado.

| Célula | O que faz |
|---|---|
| 1 | Instala o TerraTorch 1.2.11 |
| 2 | Monta o Drive, clona o repositório, confere o plano, a chave, os pesos e o manifesto da semana 6 |
| 3 | Confere o backbone |
| 4 | Baixa o teste da fonte e a Bolívia e confere os rótulos com os da semana 6 |
| 5 | Lê as 242 janelas do núcleo do T2 e confere os pixels pontuados |
| 6 | Monta o modelo (código da semana 6) e o codificador pré-treinado |
| 7 | Calcula o CKA das 11 execuções (o que já foi gravado é pulado) |
| 8 | Remonta o Drive, confere os arquivos pelo SHA-256 e grava o manifesto |

**Saída** em `resultados/semana11/e7_cka`: `cka_<regime>_s<semente>.json`, `manifesto_sha256.csv` e `info_e7.json`.

Abra o Colab na conta do mestrado, **com GPU T4**. Conte com cerca de 1 hora. Se a sessão cair, rode de novo a partir da célula 1 (ou da 2, se o TerraTorch ainda estiver instalado). Se alguma célula der erro, pare e guarde a saída inteira. Depois da célula 8, me mande `info_e7.json`.

In [ ]:
# Célula 1 de 8: instalação
!pip install -q "terratorch[peft]==1.2.11" "torchgeo==0.7.1" "peft==0.21.2"
!pip uninstall -y -q torchao

from importlib.metadata import version
for pacote in ('terratorch', 'torchgeo', 'peft', 'lightning', 'torch', 'numpy'):
    print(f'{pacote:11s} {version(pacote)}')
assert version('terratorch') == '1.2.11', 'Versão do TerraTorch diferente da fixada.'

import sys
if 'numpy' in sys.modules and sys.modules['numpy'].__version__ != version('numpy'):
    print(f"\nREINICIE A SESSÃO: o NumPy carregado é o {sys.modules['numpy'].__version__} e o instalado é o {version('numpy')}.")
    print('Menu Ambiente de execução > Reiniciar sessão (não precisa excluir o ambiente) e rode a partir da célula 2.')
else:
    print('\nNumPy carregado e instalado coincidem. Pode seguir para a célula 2.')

In [ ]:
# Célula 2 de 8: Drive, repositório, plano, chave, pesos e manifesto da semana 6
import os, re, json, time, hashlib, shutil, glob, gc, csv, subprocess, sys
import numpy as np

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

S5 = f'{RAIZ}/resultados/semana5'
MARGENS6 = f'{RAIZ}/resultados/semana6/margens'
CHAVE6 = f'{RAIZ}/resultados/semana6_chave/chave_codigos.json'
HASHES_S1F11 = f'{RAIZ}/resultados/semana6/entradas_sen1floods11_sha256.csv'
JANELAS = f'{RAIZ}/dados/worldfloods_v2/janelas'
SAIDA = f'{RAIZ}/resultados/semana11/e7_cka'           # pasta fixa: uma sessão nova continua de onde a anterior parou
LOCAL = '/content/e7'
assert not os.path.exists(f'{SAIDA}/manifesto_sha256.csv'), (
    'A pasta de saída já tem uma rodada completa (manifesto_sha256.csv). Nada é gravado por cima dela.')
for d in (SAIDA, LOCAL):
    os.makedirs(d, exist_ok=True)
for resto in glob.glob(f'{SAIDA}/parcial_*'):
    os.remove(resto)                                    # cópia interrompida de uma sessão anterior

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

def esperar(caminho, limite=180):
    t0 = time.time()
    while not os.path.exists(caminho):
        if time.time() - t0 > limite:
            raise FileNotFoundError(f'{caminho} não apareceu no Drive em {limite} s. Pare e guarde esta saída.')
        time.sleep(5)

def trazer(no_drive, sha_esperado, pasta=LOCAL):
    # Copia do Drive para o disco local e confere o SHA-256; se não conferir, remonta o Drive uma vez e copia de novo.
    local = f'{pasta}/{os.path.basename(no_drive)}'
    for tentativa in (1, 2):
        esperar(no_drive)
        if not (os.path.exists(local) and sha256_arquivo(local) == sha_esperado):
            shutil.copyfile(no_drive, local + '.parcial')
            os.replace(local + '.parcial', local)
        sha = sha256_arquivo(local)
        if sha == sha_esperado:
            return local
        if tentativa == 1:
            print(f'   {os.path.basename(no_drive)}: SHA-256 diferente do registro; remontando o Drive e copiando de novo.')
            drive.flush_and_unmount()
            time.sleep(10)
            drive.mount('/content/drive', force_remount=True)
    raise ValueError(f'{no_drive}: SHA-256 {sha[:16]}..., o registro diz {sha_esperado[:16]}... Pare e guarde esta saída.')

def gravar_no_drive(local, nome):
    parcial = f'{SAIDA}/parcial_{nome}'
    shutil.copyfile(local, parcial)
    os.replace(parcial, f'{SAIDA}/{nome}')

def carregar_npz(caminho):
    with np.load(caminho, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO_GIT = '/content/repo'
if not os.path.isdir(REPO_GIT):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_GIT], check=True)
else:
    subprocess.run(['git', '-C', REPO_GIT, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO_GIT, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()

tabela = open(f'{REPO_GIT}/analysis/README.md', encoding='utf-8').read()
registrados = dict(re.findall(r'\| `([a-z_]+\.py)` \| `([0-9a-f]{64})` \|', tabela))
assert set(registrados) == {'analise_congelada.py', 'nucleo.py', 'reamostragem.py', 'categorias.py'}, registrados.keys()
for nome, sha in registrados.items():
    assert sha256_arquivo(f'{REPO_GIT}/analysis/{nome}') == sha, f'{nome} é diferente do congelado. Não rode.'
data_plano = subprocess.run(['git', '-C', REPO_GIT, 'log', '--diff-filter=A', '-1', '--format=%cI %h', '--',
                             'prereg/plano_exploratorio_2.md'], capture_output=True, text=True, check=True).stdout.strip()
assert data_plano, 'prereg/plano_exploratorio_2.md não está no repositório.'
sys.path.insert(0, f'{REPO_GIT}/analysis/exploratoria')
import e7_cka as E7
SHA_E7 = sha256_arquivo(f'{REPO_GIT}/analysis/exploratoria/e7_cka.py')

CHAVE = json.load(open(CHAVE6))
assert CHAVE == {'8tjg': 'R1', '4m9f': 'R2', '6wke': 'R3'}, 'A chave da semana 6 não é a registrada em prereg/desvios.md.'
CODIGO = {r: c for c, r in CHAVE.items()}

CORRIDAS = [('R1', 1)] + [(r, s) for r in ('R2', 'R3') for s in (1, 2, 3, 4, 5)]
PESOS = {}
for regime, semente in CORRIDAS:
    boas = []
    for tentativa in (1, 2):
        registro = f'{S5}/exec_{regime}_s{semente}_t{tentativa}.json'
        if os.path.exists(registro) and json.load(open(registro)).get('situacao') == 'ok':
            boas.append(registro)
    pesos = f'{S5}/pesos_{regime}_s{semente}.pt'
    assert len(boas) == 1 and os.path.exists(pesos), f'Faltam arquivos do treino {regime} s{semente} em {S5}.'
    PESOS[(regime, semente)] = dict(pesos=pesos, bytes=os.path.getsize(pesos))

MANIFESTO6 = {l['arquivo']: l['sha256'] for l in csv.DictReader(open(f'{MARGENS6}/manifesto_sha256.csv', newline=''))}
assert len(MANIFESTO6) == 64, f'O manifesto da semana 6 tem {len(MANIFESTO6)} arquivos, esperado 64.'
print(f'Repositório no commit {commit[:7]}. Scripts congelados conferem. Plano adicionado em {data_plano}.')
print(f'{len(PESOS)} execuções a processar. Saída em {SAIDA.replace(MEUDRIVE, "MyDrive")}')

In [ ]:
# Célula 3 de 8: backbone
from huggingface_hub import hf_hub_download

REPO = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL'
ARQUIVO = 'Prithvi_EO_V2_300M_TL.pt'
REVISAO = '63adbd39c271da4c42f447e69b1a7c91a338cdc9'
SHA_BACKBONE = '3629cedfbb350faafcb0dac902ae0d3c927e25ce8d9e0024aa1276ec66956ddb'

no_drive = f'{RAIZ}/modelo_fixado/backbone/{ARQUIVO}'
if not os.path.exists(no_drive):
    print('Backbone ausente no Drive; baixando na revisão fixada.')
    hf_hub_download(REPO, ARQUIVO, revision=REVISAO, local_dir=os.path.dirname(no_drive))
BACKBONE_PT = f'/content/{ARQUIVO}'
if not (os.path.exists(BACKBONE_PT) and sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE):
    shutil.copy(no_drive, BACKBONE_PT)
assert sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE, 'sha256 do backbone diferente do registrado em modelo.md.'
print(f'Backbone conferido ({SHA_BACKBONE[:8]}…), cópia local em {BACKBONE_PT}')

In [ ]:
# Célula 4 de 8: teste da fonte e Bolívia, com os rótulos conferidos com os da semana 6
import urllib.request, concurrent.futures as cf
import rasterio

BUCKET = 'https://storage.googleapis.com/sen1floods11/v1.1/'
DADOS = '/content/sen1floods11'
S2 = f'{DADOS}/v1.1/data/flood_events/HandLabeled/S2Hand'
LB = f'{DADOS}/v1.1/data/flood_events/HandLabeled/LabelHand'
for d in (S2, LB):
    os.makedirs(d, exist_ok=True)
TAM = 512
IDX_BANDAS = [1, 2, 3, 8, 11, 12]          # B2, B3, B4, B8A, B11, B12 nos 13 canais do Sen1Floods11
ESCALA = np.float32(0.0001)
N_ESPERADO = {'teste': 90, 'bolivia': 15, 't2': 561}

hashes = {l['arquivo']: l['sha256'] for l in csv.DictReader(open(HASHES_S1F11, newline=''))}
ROTULOS = {}
for conj in ('teste', 'bolivia', 't2'):
    nome = f'labels_{conj}.npz'
    local = trazer(f'{MARGENS6}/{nome}', MANIFESTO6[nome])
    ROTULOS[conj] = carregar_npz(local)
    os.remove(local)
    assert len(ROTULOS[conj]['chaves']) == N_ESPERADO[conj], f'{nome}: {len(ROTULOS[conj]["chaves"])} unidades.'

def baixar(trabalho):
    url, destino = trabalho
    nome = os.path.basename(destino)
    if not (os.path.exists(destino) and sha256_arquivo(destino) == hashes[nome]):
        for tentativa in range(4):
            try:
                urllib.request.urlretrieve(url, destino + '.parcial')
                os.replace(destino + '.parcial', destino)
                break
            except Exception:
                time.sleep(2 * (tentativa + 1))
    assert sha256_arquivo(destino) == hashes[nome], f'{nome} diferente do hash gravado na semana 6.'
    return nome

def preencher(x, valor):
    x = np.ma.filled(x.astype(np.float32), np.nan)
    return np.where(np.isnan(x), np.float32(valor), x).astype(np.float32)

def ler_chip(chave):
    with rasterio.open(f'{S2}/{chave}_S2Hand.tif') as s:
        assert s.count == 13 and (s.height, s.width) == (TAM, TAM), f'{chave}: imagem'
        x = preencher(s.read(masked=True), 0.0)[IDX_BANDAS]
    with rasterio.open(f'{LB}/{chave}_LabelHand.tif') as s:
        assert s.count == 1 and (s.height, s.width) == (TAM, TAM), f'{chave}: rótulo'
        y = preencher(s.read(1, masked=True), -1.0).astype(np.int16)
    return x, y

CONJ = {}
for conj, grupo_fixo in (('teste', 'teste'), ('bolivia', 'Bolivia')):
    r = ROTULOS[conj]
    chaves = [str(c) for c, t in zip(r['chaves'], r['tamanhos']) if t > 0]
    trabalhos = []
    for chave in chaves:
        trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/S2Hand/{chave}_S2Hand.tif', f'{S2}/{chave}_S2Hand.tif'))
        trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/LabelHand/{chave}_LabelHand.tif', f'{LB}/{chave}_LabelHand.tif'))
    with cf.ThreadPoolExecutor(16) as pool:
        list(pool.map(baixar, trabalhos))
    X = np.empty((len(chaves), len(IDX_BANDAS), TAM, TAM), np.float32)
    Y = np.empty((len(chaves), TAM, TAM), np.int16)
    for i, c in enumerate(chaves):
        X[i], Y[i] = ler_chip(c)
    pos = {str(c): i for i, c in enumerate(r['chaves'])}
    inicio = np.concatenate([[0], np.cumsum(r['tamanhos'])])
    for i, c in enumerate(chaves):
        a, b = inicio[pos[c]], inicio[pos[c] + 1]
        pont = Y[i] >= 0
        assert int(pont.sum()) == b - a and np.array_equal(Y[i][pont].astype(np.uint8), r['rotulo'][a:b]), (
            f'{c}: rótulo diferente do da semana 6.')
    CONJ[conj] = dict(chaves=chaves, bruto=X, ys=Y, grupos=[grupo_fixo] * len(chaves))
    print(f'{conj}: {len(chaves)} unidades com pixel pontuado, conferidas pelos hashes e pelos rótulos da semana 6.')

In [ ]:
# Célula 5 de 8: as 242 janelas do núcleo do T2
def ler_csv(caminho):
    with open(caminho, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

caminho_janelas = f'{REPO_GIT}/splits/t2_janelas.csv'
assert sha256_arquivo(caminho_janelas) == '3c21970835f2d43cd436be9616c451ebbff5fcea138b7401d3867f0901643b2c', (
    'splits/t2_janelas.csv diferente do registrado.')
nucleo = {r['map_id'] for r in ler_csv(f'{REPO_GIT}/splits/t2_eventos.csv') if r['core'] == 'True'}
linhas = sorted((r for r in ler_csv(caminho_janelas) if r['map_id'] in nucleo and r['kept_primary'] == 'True'),
                key=lambda r: r['window_id'])
contagem = {}
for r in linhas:
    contagem[r['event']] = contagem.get(r['event'], 0) + 1
assert contagem == {'EMSR333': 2, 'EMSR342': 92, 'EMSR347': 141, 'EMSR438': 7}, contagem

r6 = ROTULOS['t2']
pos = {str(c): i for i, c in enumerate(r6['chaves'])}
X = np.empty((len(linhas), len(IDX_BANDAS), TAM, TAM), np.int16)
Y = np.empty((len(linhas), TAM, TAM), np.int16)
for i, r in enumerate(linhas):
    jid = r['window_id']
    with rasterio.open(f'{JANELAS}/S2/{jid}.tif') as s:
        assert s.count == 6 and (s.height, s.width) == (TAM, TAM) and s.dtypes[0] == 'int16', f'{jid}: imagem'
        X[i] = s.read()
    with rasterio.open(f'{JANELAS}/label/{jid}.tif') as s:
        assert s.count == 1 and (s.height, s.width) == (TAM, TAM), f'{jid}: rótulo'
        Y[i] = preencher(s.read(1, masked=True), -1.0).astype(np.int16)
    assert int((Y[i] >= 0).sum()) == int(r6['tamanhos'][pos[jid]]) == int(r['scored_pixels']), f'{jid}: pixels pontuados.'
CONJ['t2'] = dict(chaves=[r['window_id'] for r in linhas], bruto=X, ys=Y, grupos=[r['event'] for r in linhas])
print(f't2: {len(linhas)} janelas do núcleo lidas; pixels pontuados iguais aos da semana 6 e da tabela de janelas.')

In [ ]:
# Célula 6 de 8: modelo (código da semana 6) e codificador pré-treinado
import warnings, contextlib, io, logging, gc
import torch
from albumentations.pytorch import ToTensorV2
from importlib.metadata import version
try:
    from terratorch.tasks import SemanticSegmentationTask
    from terratorch.datamodules import Sen1Floods11NonGeoDataModule
except ModuleNotFoundError as erro:
    raise RuntimeError('TerraTorch não está instalado nesta sessão. Rode a célula 1 e confira que ela imprime terratorch 1.2.11.') from erro

warnings.filterwarnings('ignore', category=UserWarning)
assert torch.cuda.is_available(), 'Sem GPU: Ambiente de execução > Alterar tipo de ambiente de execução > T4.'
DISPOSITIVO = 'cuda'
BANDAS = ['BLUE', 'GREEN', 'RED', 'NIR_NARROW', 'SWIR_1', 'SWIR_2']
ESPERADO_M = {'R1': 15.1, 'R2': 20.6, 'R3': 319.0}
TOLERANCIA_M = 0.1
LR = {'R1': 5e-4, 'R2': 5e-4, 'R3': 5e-5}
EPOCAS = 50
LOTE_INF = 8
CAMADAS = E7.CAMADAS
N_TOKENS = (TAM // 16) ** 2

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

def datamodule():
    para_tensor = lambda: ToTensorV2(transpose_mask=False)
    return Sen1Floods11NonGeoDataModule(
        data_root=DADOS, batch_size=LOTE_INF, num_workers=0, bands=BANDAS,
        train_transform=[para_tensor()], val_transform=[para_tensor()], test_transform=[para_tensor()],
        drop_last=False, constant_scale=0.0001, no_data_replace=0.0, no_label_replace=-1, use_metadata=False)

LORA = dict(method='LORA', replace_qkv='qkv',
            peft_config_kwargs=dict(target_modules=['qkv.q_linear', 'qkv.v_linear', 'mlp.fc1', 'mlp.fc2'],
                                    lora_alpha=16, r=16))

def tarefa(regime):
    args = dict(
        backbone='prithvi_eo_v2_300_tl', backbone_pretrained=True, backbone_ckpt_path=BACKBONE_PT,
        backbone_bands=BANDAS,
        necks=[{'name': 'SelectIndices', 'indices': [5, 11, 17, 23]}, {'name': 'ReshapeTokensToImage'}],
        decoder='UperNetDecoder', decoder_channels=256, decoder_scale_modules=True,
        head_dropout=0.1, num_classes=2, rescale=True)
    if regime == 'R2':
        args['peft_config'] = LORA
    t = SemanticSegmentationTask(
        model_factory='EncoderDecoderFactory', model_args=args, loss='ce', ignore_index=-1,
        lr=LR[regime], optimizer='AdamW', optimizer_hparams={'weight_decay': 0.05},
        scheduler='CosineAnnealingLR', scheduler_hparams={'T_max': EPOCAS, 'eta_min': 0},
        freeze_backbone=(regime == 'R1'), plot_on_val=0)
    t.configure_models()
    treinaveis = sum(p.numel() for p in t.parameters() if p.requires_grad)
    if regime == 'R2':
        n_lora = sum(1 for n, _ in t.model.encoder.named_parameters() if 'lora_A' in n)
        assert n_lora == 96, 'Número de módulos LoRA diferente do esperado.'
    assert abs(treinaveis / 1e6 - ESPERADO_M[regime]) <= TOLERANCIA_M, 'Parâmetros treináveis fora do esperado.'
    return t

DM = datamodule()

@contextlib.contextmanager
def em_silencio():
    logging.disable(logging.CRITICAL)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)

def carregar_modelo(regime, semente):
    with em_silencio():
        t = tarefa(regime)
        t.model.to(DISPOSITIVO)
        gc.collect()
        pesos = torch.load(PESOS[(regime, semente)]['pesos'], map_location=DISPOSITIVO)
        do_modelo = set(t.model.state_dict().keys())
        esperado = do_modelo if regime != 'R1' else {k for k in do_modelo if not k.startswith('encoder.')}
        assert set(pesos.keys()) == esperado, 'Os tensores salvos não coincidem com os do modelo.'
        resultado = t.model.load_state_dict(pesos, strict=False)
        assert not resultado.unexpected_keys, 'Tensores salvos sem lugar no modelo.'
        assert set(resultado.missing_keys) == (do_modelo - esperado), 'Tensores do modelo sem peso.'
        modelo = t.model.to(DISPOSITIVO).float().eval()
        for p in modelo.parameters():
            p.requires_grad_(False)
        del t, pesos
    gc.collect()
    return modelo

with em_silencio():
    _t = tarefa('R1')
    ENC_PRE = _t.model.encoder.to(DISPOSITIVO).float().eval()
    for p in ENC_PRE.parameters():
        p.requires_grad_(False)
    del _t
gc.collect()

def entrada(conj, i):
    x = torch.from_numpy(conj['bruto'][i:i + LOTE_INF].astype(np.float32) * ESCALA).to(DISPOSITIVO)
    y = torch.from_numpy(conj['ys'][i:i + LOTE_INF].astype(np.int64)).to(DISPOSITIVO)
    x = DM.aug({'image': x, 'mask': y})['image']
    assert x.dtype == torch.float32 and x.ndim == 4 and x.shape[1] == len(BANDAS), tuple(x.shape)
    return x

def tokens(encoder, x):
    # Saídas dos blocos 5, 11, 17 e 23, só os tokens de recorte (o token de classe, se houver, sai).
    saidas = encoder(x)
    assert isinstance(saidas, (list, tuple)) and len(saidas) == 24, f'O codificador devolveu {type(saidas)} de tamanho {len(saidas)}.'
    out = {}
    for c in CAMADAS:
        f = saidas[c]
        assert f.ndim == 3 and f.shape[1] in (N_TOKENS, N_TOKENS + 1), f'Bloco {c}: forma {tuple(f.shape)}'
        out[c] = f[:, -N_TOKENS:].float()
    return out

print('Definições carregadas e codificador pré-treinado montado. Nada foi inferido ainda.')

In [ ]:
# Célula 7 de 8: CKA das 11 execuções
import psutil
LIMITE_RAM_PCT = 62
LIMITE_MARGEM = 1e-3
GRUPOS = E7.GRUPOS

def somas_gpu(d):
    return {'n': 0, 'sx': torch.zeros(d, dtype=torch.float64, device=DISPOSITIVO),
            'sy': torch.zeros(d, dtype=torch.float64, device=DISPOSITIVO),
            'sxx': torch.zeros((d, d), dtype=torch.float64, device=DISPOSITIVO),
            'syy': torch.zeros((d, d), dtype=torch.float64, device=DISPOSITIVO),
            'sxy': torch.zeros((d, d), dtype=torch.float64, device=DISPOSITIVO)}

def acumular_gpu(s, xg, yg):
    xg, yg = xg.double(), yg.double()
    s['n'] += xg.shape[0]
    s['sx'] += xg.sum(0); s['sy'] += yg.sum(0)
    s['sxx'] += xg.T @ xg; s['syy'] += yg.T @ yg; s['sxy'] += xg.T @ yg

def para_numpy(s):
    return {k: (v if k == 'n' else v.cpu().numpy()) for k, v in s.items()}

margens_bolivia = {}
for regime, semente in CORRIDAS:
    nome = f'cka_{regime}_s{semente}.json'
    if os.path.exists(f'{SAIDA}/{nome}'):
        print(f'{regime} s{semente}: já feita.')
        continue
    pct = psutil.virtual_memory().percent
    if pct > LIMITE_RAM_PCT:
        raise RuntimeError(f'PAREI: RAM em {pct:.0f}%. Reinicie a sessão e rode da célula 2 em diante; o que está feito é pulado.')
    t0 = time.time()
    cod = CODIGO[regime]
    nome6 = f'run_{cod}_s{semente}_bolivia.npz'
    local6 = trazer(f'{MARGENS6}/{nome6}', MANIFESTO6[nome6])
    z6 = carregar_npz(local6)
    os.remove(local6)
    modelo = carregar_modelo(regime, semente)
    somas, d = None, None
    simples = []
    with torch.no_grad(), torch.autocast(device_type=DISPOSITIVO, enabled=False):
        for conj in ('teste', 'bolivia', 't2'):
            c = CONJ[conj]
            for i in range(0, len(c['ys']), LOTE_INF):
                x = entrada(c, i)
                tp = tokens(ENC_PRE, x)
                tf = tokens(modelo.encoder, x)
                if somas is None:
                    d = tp[CAMADAS[0]].shape[-1]
                    somas = {(g, k): somas_gpu(d) for g in GRUPOS for k in CAMADAS}
                for j in range(x.shape[0]):
                    g = c['grupos'][i + j]
                    for k in CAMADAS:
                        acumular_gpu(somas[(g, k)], tp[k][j], tf[k][j])
                if conj == 'bolivia':
                    z = modelo(x).output.float()
                    marg = (z[:, 1] - z[:, 0]).cpu().numpy()
                    for j in range(x.shape[0]):
                        simples.append(marg[j][c['ys'][i + j] >= 0].astype(np.float32))
                del tp, tf
    m_bol = np.concatenate(simples)
    assert m_bol.size == z6['margem'].size, f'{regime} s{semente}: margens da Bolívia com tamanho errado.'
    dif = float(np.max(np.abs(m_bol - z6['margem'])))
    assert dif <= LIMITE_MARGEM, (f'{regime} s{semente}: a margem da Bolívia difere da gravada na semana 6 em {dif:.2e}. '
                                  'Nada foi gravado. Pare e guarde esta saída.')
    cka, tokens_grupo = {}, {}
    for g in GRUPOS + ('alvo',):
        cka[g] = {}
        for k in CAMADAS:
            if g == 'alvo':
                s = E7.juntar([para_numpy(somas[(e, k)]) for e in GRUPOS if e != 'teste'])
            else:
                s = para_numpy(somas[(g, k)])
            cka[g][str(k)] = E7.cka_de_somas(**s)
            tokens_grupo[g] = int(s['n'])
    menor = min(v for gg in cka.values() for v in gg.values())
    if regime == 'R1':
        assert menor >= E7.MIN_CKA_R1, f'Conferência do R1 falhou: CKA mínimo {menor:.6f}. Pare e guarde esta saída.'
    rec = {'regime': regime, 'semente': semente, 'codigo': cod, 'camadas': list(CAMADAS), 'cka': cka,
           'tokens': tokens_grupo, 'dimensao': d, 'diferenca_margem_bolivia': dif,
           'gpu': torch.cuda.get_device_name(0), 'segundos': round(time.time() - t0), 'commit_repositorio': commit}
    json.dump(rec, open(f'{LOCAL}/{nome}', 'w'), indent=1)
    gravar_no_drive(f'{LOCAL}/{nome}', nome)
    print(f'{regime} s{semente}: CKA no alvo, bloco 23 = {cka["alvo"]["23"]:.4f}; margem da Bolívia confere ({dif:.1e}); '
          f'{(time.time() - t0) / 60:.1f} min')
    del modelo, somas
    gc.collect(); torch.cuda.empty_cache()
print('\nGuarde a saída desta célula.')

In [ ]:
# Célula 8 de 8: conferência final
print('Gravando o Drive e conferindo de novo; pode levar alguns minutos.')
drive.flush_and_unmount()
time.sleep(10)
drive.mount('/content/drive', force_remount=True)

linhas, registros = [], []
for regime, semente in CORRIDAS:
    nome = f'cka_{regime}_s{semente}.json'
    esperar(f'{SAIDA}/{nome}')
    if os.path.exists(f'{LOCAL}/{nome}'):
        assert json.load(open(f'{SAIDA}/{nome}')) == json.load(open(f'{LOCAL}/{nome}')), (
            f'{nome}: o arquivo do Drive difere do gravado nesta sessão.')
    registros.append(json.load(open(f'{SAIDA}/{nome}')))
    linhas.append([nome, os.path.getsize(f'{SAIDA}/{nome}'), sha256_arquivo(f'{SAIDA}/{nome}')])
with open(f'{SAIDA}/manifesto_sha256.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['arquivo', 'bytes', 'sha256'])
    w.writerows(linhas)
INFO = {'analise': 'exploratoria E7 (prereg/plano_exploratorio_2.md)', 'plano_adicionado': data_plano,
        'commit_repositorio': commit, 'e7_sha256': SHA_E7, 'congelados_sha256': registrados,
        'corridas': [f'{r}_s{s}' for r, s in CORRIDAS], 'camadas': list(CAMADAS),
        'grupos': {g: n for g, n in registros[0]['tokens'].items()},
        'cka_alvo_bloco23': {f"{r['regime']}_s{r['semente']}": r['cka']['alvo']['23'] for r in registros},
        'diferenca_margem_bolivia_maxima': max(r['diferenca_margem_bolivia'] for r in registros),
        'backbone_sha256': SHA_BACKBONE, 'precisao': 'float32 na inferencia, float64 nas somas do CKA',
        'versoes': {q: version(q) for q in ('terratorch', 'torchgeo', 'peft', 'torch', 'numpy')},
        'gpu': torch.cuda.get_device_name(0), 'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA}/info_e7.json', 'w'), indent=2, ensure_ascii=False)
print(f'{len(linhas)} arquivos de CKA conferidos depois de remontar o Drive.')
print('Manifesto e info_e7.json gravados em resultados/semana11/e7_cka.')